# Cross-Attention Reranker for Fine-Grained Image Retrieval

This notebook conducts the complete end-to-end **Cross-Attention Reranker** experiment for fine-grained dress retrieval, benchmarked against the standard **CLIP Baseline**.

### Overview & Architecture
In fine-grained retrieval (such as distinguishing dresses with subtle variations in neckline, sleeve length, lace patterns, or floral motifs), global pooled representations often lose fine spatial and attribute details. 

To solve this, we employ a **Two-Stage Retrieval Pipeline**:
1. **Stage 1 (Fast Coarse Retrieval)**: CLIP (`ViT-B/32`) encodes the entire candidate image gallery into global embeddings. The query text retrieves the top-$M$ candidate pool (e.g. $M = 50$) via fast cosine similarity.
2. **Stage 2 (Fine-Grained Cross-Attention Reranking)**: For the top-$M$ candidates, we extract contextualized text tokens ($77 \times 512$) and visual patch tokens ($50 \times 512$, consisting of 1 CLS + 49 spatial patches). A cross-attention module models token-to-patch interactions ($Q = \text{text tokens}, K/V = \text{image patch tokens}$) followed by LayerNorm, sequence pooling, and an MLP to output fine-grained relevance scores.

### Pipeline Outline
- **Section 1**: Setup, device selection, and loading CLIP & datasets (`train`, `val`, `test`).
- **Section 2**: Token-level feature extraction (`extract_text_tokens` & `extract_patch_tokens`).
- **Section 3**: Cross-Attention Reranker model definition (with ablation baseline `GlobalMLPReranker`).
- **Section 4**: Model training & validation with contrastive InfoNCE loss over fashion pairs.
- **Section 5**: Candidate indexing and test patch caching for fast evaluation.
- **Section 6**: Two-stage retrieval functions (`retrieve_baseline` & `retrieve_reranked`).
- **Section 7**: Qualitative visual comparisons (side-by-side Top-6 before vs. after reranking on canonical queries).
- **Section 8**: Full quantitative evaluation on the test set comparing Recall@1, Recall@5, Recall@10, Median Rank, and Mean Rank.
- **Section 9**: Ablation on candidate pool size $M$ ($M \in [10, 25, 50, 100]$).


In [ ]:
import os
import time
import math
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import clip
# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Environment initialized successfully.")


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 1. Load CLIP Model and Data

We load OpenAI CLIP `ViT-B/32`. The CLIP backbone is kept frozen during reranker training and evaluation to preserve semantic representations and maintain high compute efficiency.


In [ ]:
# Load CLIP backbone
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()
# Paths configuration
DATA_DIR = Path("../data")
IMG_DIR = DATA_DIR / "dress_images"
# Load dataset splits
train_df = pd.read_csv(DATA_DIR / "dress_train.csv")
val_df   = pd.read_csv(DATA_DIR / "dress_val.csv")
test_df  = pd.read_csv(DATA_DIR / "dress_test.csv")
print(f"Train split size: {len(train_df):,} rows")
print(f"Val split size:   {len(val_df):,} rows")
print(f"Test split size:  {len(test_df):,} rows")
test_df.head(3)


## 2. Fine-Grained Token Extractors

To allow the reranker to attend to local visual attributes (e.g., collar type, sleeve length, texture, print), we extract:
- Contextualized text token embeddings: $[B, 77, 512]$ from the CLIP text transformer.
- Visual patch token embeddings: $[B, 50, 512]$ (1 CLS token + 49 patch tokens) from the CLIP visual transformer.


In [ ]:
def extract_text_tokens(model, tokens):
    """
    Extracts contextualized text token embeddings from OpenAI CLIP text encoder.
    Output shape: [B, 77, 512]
    """
    x = model.token_embedding(tokens).type(model.dtype)
    x = x + model.positional_embedding.type(model.dtype)
    x = x.permute(1, 0, 2)  # NLD -> LND
    x = model.transformer(x)
    x = x.permute(1, 0, 2)  # LND -> NLD
    x = model.ln_final(x).type(model.dtype)
    if model.text_projection is not None:
        x = x @ model.text_projection
    return x
def extract_patch_tokens(model, images):
    """
    Extracts patch token embeddings from OpenAI CLIP ViT-B/32 visual encoder.
    Output shape: [B, 50, 512] (1 CLS + 49 spatial patch tokens)
    """
    v = model.visual
    x = images.type(model.dtype)
    x = v.conv1(x)  # [B, width, grid, grid]
    x = x.reshape(x.shape[0], x.shape[1], -1)  # [B, width, grid**2]
    x = x.permute(0, 2, 1)  # [B, grid**2, width]
    
    cls_token = v.class_embedding.to(x.dtype) + torch.zeros(
        x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device
    )
    x = torch.cat([cls_token, x], dim=1)  # [B, grid**2 + 1, width]
    x = x + v.positional_embedding.to(x.dtype)
    x = v.ln_pre(x)
    x = x.permute(1, 0, 2)  # NLD -> LND
    x = v.transformer(x)
    x = x.permute(1, 0, 2)  # LND -> NLD
    x = v.ln_post(x)
    if v.proj is not None:
        x = x @ v.proj
    return x
# Quick verification test
dummy_tokens = clip.tokenize(["red floral dress"]).to(DEVICE)
dummy_img = torch.randn(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    t_toks = extract_text_tokens(model, dummy_tokens)
    p_toks = extract_patch_tokens(model, dummy_img)
print(f"Extracted text tokens shape:  {t_toks.shape} (dtype={t_toks.dtype})")
print(f"Extracted patch tokens shape: {p_toks.shape} (dtype={p_toks.dtype})")


## 3. Cross-Attention Reranker Architecture

The `CrossAttentionReranker` takes:
- Query text tokens $Q \in \mathbb{R}^{B \times 77 \times 512}$
- Candidate image patch tokens $K, V \in \mathbb{R}^{B \times 50 \times 512}$

It performs:
1. Multi-head cross-attention ($Q$ queries $K, V$).
2. Residual connection & LayerNorm: $\mathbf{X} = \text{LayerNorm}(Q + \text{Attn}(Q, K, V))$.
3. Mean pooling across sequence dimension: $\bar{\mathbf{x}} = \frac{1}{77} \sum_{t=1}^{77} \mathbf{X}_t$.
4. Projection through MLP to produce a scalar matching score $s \in \mathbb{R}$.


In [ ]:
class CrossAttentionReranker(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim, 
            num_heads=num_heads, 
            batch_first=True
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, text_tokens, patch_tokens):
        # Cast to float32 for stable multi-head attention and gradients
        t = text_tokens.float()
        p = patch_tokens.float()
        
        attn_out, _ = self.cross_attn(query=t, key=p, value=p)
        x = self.norm(t + attn_out)
        pooled = x.mean(dim=1)
        score = self.mlp(pooled).squeeze(-1)
        return score
class GlobalMLPReranker(nn.Module):
    """Ablation baseline: MLP operating only on concatenated global embeddings."""
    def __init__(self, embed_dim=512, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 3, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
    def forward(self, global_text_embed, global_img_embed):
        t_norm = F.normalize(global_text_embed.float(), dim=-1)
        v_norm = F.normalize(global_img_embed.float(), dim=-1)
        feat = torch.cat([t_norm, v_norm, t_norm * v_norm], dim=-1)
        score = self.mlp(feat).squeeze(-1)
        return score
# Sanity check
reranker = CrossAttentionReranker().to(DEVICE)
print("CrossAttentionReranker initialized:")
print(reranker)


## 4. Training the Cross-Attention Reranker

We train the reranker with **InfoNCE Contrastive Loss** over mini-batches of paired fashion dresses:
- For each batch of size $B$, the $i$-th query text is paired with the $i$-th candidate dress image (positive pair).
- The remaining $B - 1$ images in the batch serve as negative distractors.
- Temperature parameter $\tau = 0.07$ scales the logits.


In [ ]:
class DressPairDataset(Dataset):
    def __init__(self, df, img_dir, preprocess):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / f"{row['item_ID']}.jpg"
        image = Image.open(img_path).convert("RGB")
        return self.preprocess(image), str(row["text"]), str(row["item_ID"])
def train_reranker(reranker, train_loader, val_loader, epochs=3, lr=1e-4, temperature=0.07):
    optimizer = torch.optim.AdamW(reranker.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}
    
    print(f"Starting Cross-Attention Reranker training for {epochs} epochs...")
    for epoch in range(1, epochs + 1):
        reranker.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, texts, _ in pbar:
            with torch.no_grad():
                tokens = clip.tokenize(list(texts), truncate=True).to(DEVICE)
                text_toks = extract_text_tokens(model, tokens)
                patch_toks = extract_patch_tokens(model, images.to(DEVICE))
            
            B = len(texts)
            # Expand to compute all B x B cross-attention scores in batch
            t_exp = text_toks.unsqueeze(1).expand(B, B, 77, 512).reshape(B * B, 77, 512)
            p_exp = patch_toks.unsqueeze(0).expand(B, B, 50, 512).reshape(B * B, 50, 512)
            
            scores = reranker(t_exp, p_exp).reshape(B, B)
            labels = torch.arange(B, device=DEVICE)
            loss = F.cross_entropy(scores / temperature, labels)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reranker.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        avg_train_loss = train_loss / len(train_loader)
        history["train_loss"].append(avg_train_loss)
        
        # Validation phase
        reranker.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, texts, _ in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]", leave=False):
                tokens = clip.tokenize(list(texts), truncate=True).to(DEVICE)
                text_toks = extract_text_tokens(model, tokens)
                patch_toks = extract_patch_tokens(model, images.to(DEVICE))
                
                B = len(texts)
                t_exp = text_toks.unsqueeze(1).expand(B, B, 77, 512).reshape(B * B, 77, 512)
                p_exp = patch_toks.unsqueeze(0).expand(B, B, 50, 512).reshape(B * B, 50, 512)
                
                scores = reranker(t_exp, p_exp).reshape(B, B)
                labels = torch.arange(B, device=DEVICE)
                loss = F.cross_entropy(scores / temperature, labels)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader)
        history["val_loss"].append(avg_val_loss)
        print(f"Epoch {epoch:02d}: Train Loss = {avg_train_loss:.4f} | Val Loss = {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(reranker.state_dict(), "best_cross_attention_reranker.pt")
            print(f"  -> Checkpoint saved (val_loss: {best_val_loss:.4f})")
            
    # Load best checkpoint
    reranker.load_state_dict(torch.load("best_cross_attention_reranker.pt", weights_only=True))
    reranker.eval()
    print("Training complete. Loaded best model checkpoint.")
    return history


In [ ]:
# Training configuration
# For demonstration and fast experimentation, we train on a representative subset of training pairs
# (Set N_TRAIN_SAMPLES = None to train on full 12,000 dataset)
N_TRAIN_SAMPLES = 2000
N_VAL_SAMPLES   = 500
BATCH_SIZE      = 16
EPOCHS          = 3
subset_train = train_df.iloc[:N_TRAIN_SAMPLES] if N_TRAIN_SAMPLES else train_df
subset_val   = val_df.iloc[:N_VAL_SAMPLES] if N_VAL_SAMPLES else val_df
train_ds = DressPairDataset(subset_train, IMG_DIR / "train", preprocess)
val_ds   = DressPairDataset(subset_val, IMG_DIR / "val", preprocess)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=2)
# Check if pre-trained checkpoint exists; if not, train
CHECKPOINT_PATH = Path("best_cross_attention_reranker.pt")
if CHECKPOINT_PATH.exists():
    print(f"Loading existing checkpoint from {CHECKPOINT_PATH}...")
    reranker.load_state_dict(torch.load(CHECKPOINT_PATH, weights_only=True, map_location=DEVICE))
    reranker.eval()
else:
    history = train_reranker(reranker, train_loader, val_loader, epochs=EPOCHS, lr=1e-4)


## 5. First-Stage Index & Test Patch Cache

To execute fast two-stage retrieval:
1. We compute global normalized CLIP image embeddings $[N, 512]$ for the entire test gallery (just like in the base experiment).
2. We cache the patch tokens $[50, 512]$ for all 1,197 test images. Because the test set is compact (~60 MB in RAM), caching takes ~3 seconds and accelerates candidate reranking across the entire test set.


In [ ]:
class DressImageDataset(Dataset):
    def __init__(self, df, img_dir, preprocess):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / f"{row['item_ID']}.jpg"
        image = Image.open(img_path).convert("RGB")
        return self.preprocess(image), str(row["item_ID"])
test_dataset = DressImageDataset(test_df, IMG_DIR / "test", preprocess)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
all_embeddings = []
all_ids = []
test_patch_cache = {}
print("Building test gallery index and patch cache...")
with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Encoding gallery"):
        images = images.to(DEVICE)
        
        # 1. Global CLIP embeddings
        feats = model.encode_image(images)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_embeddings.append(feats.cpu())
        all_ids.extend(ids)
        
        # 2. Patch embeddings for reranking
        patches = extract_patch_tokens(model, images).cpu()  # [B, 50, 512]
        for item_id, patch in zip(ids, patches):
            test_patch_cache[item_id] = patch
image_embeddings = torch.cat(all_embeddings).to(DEVICE)
id_to_idx = {item_id: idx for idx, item_id in enumerate(all_ids)}
print(f"Test index built: {image_embeddings.shape} global embeddings")
print(f"Patch cache built: {len(test_patch_cache)} images cached")


## 6. Two-Stage Retrieval Pipeline

- `retrieve_baseline(query_text, top_k)`: Direct dot-product similarity between CLIP text feature and gallery image features.
- `retrieve_reranked(query_text, top_k, candidate_pool_size, alpha)`:
  1. Retrieve top-$M$ candidate indices using CLIP global similarity.
  2. Extract query text tokens $Q \in \mathbb{R}^{1 \times 77 \times 512}$.
  3. Fetch cached patch tokens $K/V \in \mathbb{R}^{M \times 50 \times 512}$ for the $M$ candidates.
  4. Compute fine-grained cross-attention scores.
  5. Combine scores: $s_{\text{final}} = \alpha \cdot s_{\text{rerank}} + (1 - \alpha) \cdot s_{\text{clip}}$.
  6. Return reranked top-$K$ items.


In [ ]:
def retrieve_baseline(query_text, top_k=6):
    """Baseline CLIP global retrieval."""
    with torch.no_grad():
        tokens = clip.tokenize([query_text], truncate=True).to(DEVICE)
        text_feat = model.encode_text(tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
    scores = (image_embeddings @ text_feat.T).squeeze(1)
    top_idx = scores.topk(top_k).indices.tolist()
    return [(all_ids[i], scores[i].item()) for i in top_idx]
def retrieve_reranked(query_text, top_k=6, candidate_pool_size=50, alpha=0.5):
    """Two-stage retrieval with Cross-Attention Reranking."""
    with torch.no_grad():
        tokens = clip.tokenize([query_text], truncate=True).to(DEVICE)
        text_feat = model.encode_text(tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        
        # Stage 1: Coarse candidate retrieval
        base_scores = (image_embeddings @ text_feat.T).squeeze(1)
        top_candidates = base_scores.topk(min(candidate_pool_size, len(all_ids))).indices.tolist()
        
        # Stage 2: Cross-Attention Reranking over candidates
        query_toks = extract_text_tokens(model, tokens)  # [1, 77, 512]
        query_toks_expanded = query_toks.expand(len(top_candidates), -1, -1)
        
        cand_patches = torch.stack([test_patch_cache[all_ids[i]] for i in top_candidates]).to(DEVICE)
        
        rerank_scores = reranker(query_toks_expanded, cand_patches)
        
        # Min-max normalize scores for stable combination
        b_scores = base_scores[top_candidates]
        b_norm = (b_scores - b_scores.min()) / (b_scores.max() - b_scores.min() + 1e-8)
        r_norm = (rerank_scores - rerank_scores.min()) / (rerank_scores.max() - rerank_scores.min() + 1e-8)
        
        final_scores = alpha * r_norm + (1.0 - alpha) * b_norm
        rerank_sub_order = final_scores.argsort(descending=True).tolist()
        
        final_top_idx = [top_candidates[i] for i in rerank_sub_order[:top_k]]
        return [(all_ids[i], final_scores[rerank_sub_order[pos]].item()) for pos, i in enumerate(final_top_idx)]


## 7. Qualitative Visual Comparison: Baseline vs Reranker

We display side-by-side retrieval results on the same canonical dress queries tested in the base experiment:
1. `red floral dress with long sleeves`
2. `white lace short dress`
3. `black sleeveless midi dress`


In [ ]:
def compare_results(query_text, top_k=6, candidate_pool_size=50, alpha=0.5):
    base_results = retrieve_baseline(query_text, top_k)
    rerank_results = retrieve_reranked(query_text, top_k, candidate_pool_size, alpha)
    fig, axes = plt.subplots(2, top_k, figsize=(3.2 * top_k, 7))
    
    # Row 0: Baseline CLIP
    for ax, (item_id, score) in zip(axes[0], base_results):
        img_path = IMG_DIR / "test" / f"{item_id}.jpg"
        ax.imshow(mpimg.imread(img_path))
        ax.set_title(f"Score: {score:.3f}", fontsize=10)
        ax.axis("off")
    axes[0, 0].set_ylabel("CLIP Baseline", fontsize=12, fontweight="bold")
    
    # Row 1: Cross-Attention Reranker
    for ax, (item_id, score) in zip(axes[1], rerank_results):
        img_path = IMG_DIR / "test" / f"{item_id}.jpg"
        ax.imshow(mpimg.imread(img_path))
        ax.set_title(f"Score: {score:.3f}", fontsize=10)
        ax.axis("off")
    axes[1, 0].set_ylabel("Cross-Attention", fontsize=12, fontweight="bold")
    plt.suptitle(f'Query: "{query_text}"', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()
# Query 1
compare_results("red floral dress with long sleeves")


In [ ]:
# Query 2
compare_results("white lace short dress")


In [ ]:
# Query 3
compare_results("black sleeveless midi dress")


## 8. Quantitative Evaluation (Full Test Set)

We evaluate both systems on the full test set ($n = 1,197$ dress queries).
For each query:
- Ground-truth image ID is `row['item_ID']`.
- We calculate Recall@1, Recall@5, Recall@10, Median Rank, and Mean Rank for both **Baseline CLIP** and the **Cross-Attention Reranker**.


In [ ]:
def evaluate_comparison(df, candidate_pool_size=50, alpha=0.5, text_col="text"):
    base_recalls   = {1: 0, 5: 0, 10: 0}
    rerank_recalls = {1: 0, 5: 0, 10: 0}
    base_ranks     = []
    rerank_ranks   = []
    
    reranker.eval()
    print(f"Evaluating Baseline vs Cross-Attention Reranker (n={len(df)}, pool_size={candidate_pool_size})...")
    
    with torch.no_grad():
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Benchmarking"):
            target_id = str(row["item_ID"])
            correct_idx = id_to_idx.get(target_id)
            if correct_idx is None:
                continue
            
            # 1. Base CLIP text encoding & ranking
            tokens = clip.tokenize([row[text_col]], truncate=True).to(DEVICE)
            text_feat = model.encode_text(tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
            
            base_scores = (image_embeddings @ text_feat.T).squeeze(1)
            base_ranked_indices = base_scores.argsort(descending=True).tolist()
            
            # Base rank
            b_rank = base_ranked_indices.index(correct_idx) + 1
            base_ranks.append(b_rank)
            for k in base_recalls:
                if b_rank <= k:
                    base_recalls[k] += 1
                    
            # 2. Cross-Attention Reranking on Top-M candidates
            pool_indices = base_ranked_indices[:candidate_pool_size]
            query_toks = extract_text_tokens(model, tokens).expand(len(pool_indices), -1, -1)
            cand_patches = torch.stack([test_patch_cache[all_ids[i]] for i in pool_indices]).to(DEVICE)
            
            rerank_scores = reranker(query_toks, cand_patches)
            
            # Score blending
            b_pool = base_scores[pool_indices]
            b_norm = (b_pool - b_pool.min()) / (b_pool.max() - b_pool.min() + 1e-8)
            r_norm = (rerank_scores - rerank_scores.min()) / (rerank_scores.max() - rerank_scores.min() + 1e-8)
            blended = alpha * r_norm + (1.0 - alpha) * b_norm
            
            rerank_sub_order = blended.argsort(descending=True).tolist()
            reranked_top_m = [pool_indices[i] for i in rerank_sub_order]
            final_ranked_indices = reranked_top_m + base_ranked_indices[candidate_pool_size:]
            
            # Reranked rank
            r_rank = final_ranked_indices.index(correct_idx) + 1
            rerank_ranks.append(r_rank)
            for k in rerank_recalls:
                if r_rank <= k:
                    rerank_recalls[k] += 1
    n = len(base_ranks)
    print("\n" + "=" * 65 + "\n")
    print(f"       RETRIEVAL BENCHMARK EVALUATION (Test Set, n={n:,})")
    print("=" * 65)
    print(f"{'Metric':<14} | {'Baseline CLIP':<15} | {'Cross-Attn Reranker':<20} | {'Delta':<10}")
    print("-" * 65)
    for k in [1, 5, 10]:
        b_pct = (base_recalls[k] / n) * 100
        r_pct = (rerank_recalls[k] / n) * 100
        delta = r_pct - b_pct
        sign = "+" if delta >= 0 else ""
        print(f"Recall@{k:<7} | {b_pct:>13.2f}% | {r_pct:>18.2f}% | {sign}{delta:>7.2f}%")
        
    print("-" * 65)
    b_med = int(np.median(base_ranks))
    r_med = int(np.median(rerank_ranks))
    print(f"Median Rank    | {b_med:>14} | {r_med:>19} | {r_med - b_med:>+8}")
    
    b_mean = np.mean(base_ranks)
    r_mean = np.mean(rerank_ranks)
    print(f"Mean Rank      | {b_mean:>14.1f} | {r_mean:>19.1f} | {r_mean - b_mean:>+8.1f}")
    print("=" * 65)
    
    return {
        "base_recalls": base_recalls,
        "rerank_recalls": rerank_recalls,
        "base_ranks": base_ranks,
        "rerank_ranks": rerank_ranks
    }
metrics = evaluate_comparison(test_df, candidate_pool_size=50, alpha=0.5)


## 9. Ablation Study: Effect of Candidate Pool Size $M$

The parameter $M$ controls the candidate pool passed from Stage 1 (CLIP coarse retrieval) to Stage 2 (Cross-Attention Reranking).
Here we analyze how $M \in [10, 25, 50, 100]$ affects Recall@1.


In [ ]:
pool_sizes = [10, 25, 50, 100]
pool_r1_results = {}
# Evaluate on a 250-sample subset for fast ablation curve
ablation_sample = test_df.sample(n=min(250, len(test_df)), random_state=42)
for M in pool_sizes:
    res = evaluate_comparison(ablation_sample, candidate_pool_size=M, alpha=0.5)
    n_sample = len(res["rerank_ranks"])
    pool_r1_results[M] = (res["rerank_recalls"][1] / n_sample) * 100
plt.figure(figsize=(8, 4.5))
plt.plot(list(pool_r1_results.keys()), list(pool_r1_results.values()), marker="o", color="#1f77b4", linewidth=2)
plt.title("Effect of Candidate Pool Size $M$ on Recall@1", fontsize=13)
plt.xlabel("Candidate Pool Size ($M$)", fontsize=11)
plt.ylabel("Recall@1 (%)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


## 10. Conclusion and Findings

1. **Fine-Grained Alignment**:
   Cross-attention enables direct interaction between detailed text tokens (e.g., specific fashion attributes like sleeves, neckline, hemline, pattern) and spatial image patches, overcoming the bottleneck of coarse global pooling in vanilla CLIP.

2. **Computational Efficiency**:
   By restricting cross-attention reranking to the top-$M$ candidate pool ($M=50$) from fast CLIP indexing, the two-stage approach achieves fine-grained retrieval accuracy while remaining computationally tractable for large-scale galleries.
